In [6]:
import pandas as pd
import polars as pl
from pathlib import Path

In [7]:
raw_data_dir = Path.cwd().parent / 'data_gather' / 'raw_data'
filtered_data_dir = Path.cwd().parent / 'data_gather' / 'filtered_data'


In [9]:
trades = pl.scan_csv(raw_data_dir / 'coinbase_trades.csv').collect()
options = pl.scan_csv(raw_data_dir / 'deribit_option_vols.csv').collect()
kalshi = pl.scan_csv(raw_data_dir / 'kalshi_markets.csv').collect()
polymarket = pl.scan_csv(raw_data_dir / 'polymarket_markets.csv').collect()

In [ ]:
dfz = df.with_columns(
    pl.col("curr_time")
    .str.to_datetime(format="%Y-%m-%d %H:%M:%S%.f%#z", strict=False)
    .alias("curr_time"))

dfz.head()

NameError: name 'df' is not defined

In [10]:
class filter_data(): 
    def __init__(self
                 , trades: pl.DataFrame
                 , options: pl.DataFrame
                 , kalshi: pl.DataFrame
                 , polymarket: pl.DataFrame):
        
        self.trades = trades
        self.options = options
        self.kalshi = kalshi
        self.polymarket = polymarket
    
    def clean_trades(self, coins: list) -> pl.DataFrame:
        coin_trades = self.trades.filter(pl.col('product_id').is_in([f'{coin}-USD' for coin in coins]))
        coin_trades = coin_trades.unique(subset=['trade_id'])
        coin_trades = coin_trades.sort('trade_time', descending=False)
        coin_trades = coin_trades.drop('id').with_row_index('id')
        coin_trades = coin_trades.select('id', 'curr_time', 'product_id', 'trade_time', 'price', 'size', 'side')
        return coin_trades
    
    def clean_options(self, coins: list) -> pl.DataFrame: 
        coin_options = self.options.filter(pl.col('currency').is_in([f'{coin}' for coin in coins]))
        coin_options = coin_options.sort('curr_time', descending=False)
        coin_options = coin_options.drop('id').with_row_index('id')
        coin_options = coin_options.select('id', 'curr_time', 'instrument_name', 'expiry_datetime', 'strike', 
                                           'option_type', 'underlying_price', 'delta', 'mark_iv', 'mark_price', 
                                           'open_interest', 'volume')
        return coin_options
    
    def clean_kalshi(self, coins: list) -> pl.DataFrame: 
        coin_kalshi = self.kalshi.filter(pl.col('coin').is_in([f'{coin}' for coin in coins]))
        coin_kalshi = coin_kalshi.sort('curr_time', descending=False)
        coin_kalshi = coin_kalshi.drop('id').with_row_index('id')
        coin_kalshi = coin_kalshi.select('id', 'curr_time', 'coin', 'open_time', 'close_time', 
                                         'last_price_dollars' , 'no_ask_dollars', 'no_bid_dollars', 
                                         'yes_ask_dollars', 'yes_bid_dollars', 'floor_strike', 'volume_24h_fp')
        return coin_kalshi
    
    def clean_polymarket(self, coins: list) -> pl.DataFrame:
        coin_polymarket = self.polymarket.filter(pl.col('coin').is_in([f'{coin}' for coin in coins]))
        coin_polymarket = coin_polymarket.sort('curr_time', descending=False)
        coin_polymarket = coin_polymarket.drop('id').with_row_index('id')
        coin_polymarket = coin_polymarket.select('id', 'curr_time', 'coin', 'interval_start_unix', 'end_date', 'strike_price',
                                                 'liquidity', 'volume', 'yes_implied_price', 'no_implied_price', 'yes_buy_price',
                                                 'yes_sell_price', 'no_buy_price', 'no_sell_price')
        return coin_polymarket
    
        

In [17]:
FILTERER = filter_data(trades=trades, options=options, kalshi=kalshi, polymarket=polymarket)


In [96]:
btc_trades = FILTERER.clean_trades(['BTC'])
btc_eth_options = FILTERER.clean_options(['BTC', 'ETH'])
all_coins_kalshi = FILTERER.clean_kalshi(['BTC', 'ETH', 'XRP', 'SOL'])
all_coins_polymarket = FILTERER.clean_polymarket(['BTC', 'ETH', 'XRP', 'SOL'])

In [97]:
def _to_datetime(table: pl.DataFrame, cols: list) -> pl.DataFrame:
    for col in cols: 
        table = table.with_columns(
            pl.col(col)
            .str.to_datetime(format="%Y-%m-%d %H:%M:%S%.f%#z", strict=False)
            .alias(col)
        )
    return table


In [ ]:
btc_trades = _to_datetime(btc_trades, ['curr_time', 'trade_time'])
btc_eth_options = _to_datetime(btc_eth_options, ['curr_time', 'expiry_datetime'])
all_coins_kalshi = _to_datetime(all_coins_kalshi, ['curr_time', 'open_time', 'close_time'])
all_coins_polymarket = _to_datetime(all_coins_polymarket, ['curr_time', 'end_date'])


In [ ]:
first_event_time, last_event_time = all_coins_kalshi.select(pl.col('open_time').min(), pl.col('close_time').max()).row(0)

all_coins_kalshi = all_coins_kalshi.filter(~((pl.col("open_time") == first_event_time) | (pl.col("close_time") == last_event_time)))
all_coins_kalshi.head()
all_coins_kalshi.group_by(('open_time')).agg(pl.len()).sort('open_time')


In [ ]:

df = df.with_columns(
    pl.col("curr_time").shift(1).over("coin").alias("prev_time")
).with_columns(
    (pl.col("curr_time") - pl.col("prev_time"))
    .dt.total_seconds()
    .alias("time_diff_seconds")
)

df = df.filter(pl.col("time_diff_seconds").is_between(0.9, 1.1))
df['time_diff_seconds'].value_counts()



df


In [ ]:
def build_coin_df(df: pl.DataFrame, coin: str) -> pl.DataFrame:
    col_identifiers = ["open_time", "close_time"]

    df = (
        df
        .filter(pl.col("coin") == coin)
        .sort(col_identifiers + ["curr_time"])
        .with_columns(
            pl.col("last_price_dollars")
            .shift(-1)
            .over(col_identifiers)
            .alias("next_price_dollars_lead1")
        )
        .filter(pl.col("time_to_close") > -0.01)
    )

    rename_map = {
        col: f"{coin}_{col}"
        for col in df.columns
    }

    df = df.rename(rename_map)

    return df

df = df.with_columns((pl.col('close_time') - pl.col('curr_time')).dt.total_seconds().alias('time_to_close'))



coin_dfs = [build_coin_df(df, coin) for coin in ["BTC", "ETH", "XRP", "SOL"]]

combined = coin_dfs[0]
for next_df in coin_dfs[1:]:
    combined = combined.join(
        next_df,
        on=["open_time", "close_time"],
        how="inner",
    )

combined = combined.filter(
    pl.col("BTC_next_price_dollars_lead1").is_not_null()
)

drop_cols = [col for col in ["BTC_coin", "ETH_coin", "XRP_coin", "SOL_coin"] if col in combined.columns]
combined = combined.drop(drop_cols)

outcome_map = (
    combined
    .sort("BTC_curr_time")
    .group_by("BTC_open_time")
    .agg(
        pl.col("BTC_last_price_dollars").last().alias("last_btc_price")
    )
    .with_columns(
        (pl.col("last_btc_price") > 0.5).cast(pl.Int8).alias("outcome")
    )
    .select(["BTC_open_time", "outcome"])
)

combined = combined.join(outcome_map, on="BTC_open_time", how="left")
combined.head()


In [ ]:
btc_trades = _to_datetime(btc_trades, ['curr_time', 'trade_time'])

joined = btc_trades.join_asof(df, left_on= 'trade_time', right_on='curr_time', strategy='forward')


In [87]:
joined = joined.filter((pl.col('trade_time') >= pl.col('prev_time')) & (pl.col('trade_time') <= pl.col('curr_time')))
joined

id,curr_time,product_id,trade_time,price,size,side,id_right,curr_time_right,coin,open_time,close_time,last_price_dollars,no_ask_dollars,no_bid_dollars,yes_ask_dollars,yes_bid_dollars,floor_strike,volume_24h_fp,prev_time,time_diff_seconds
i64,"datetime[μs, UTC]",str,"datetime[μs, UTC]",f64,f64,str,u32,"datetime[μs, UTC]",str,str,str,f64,f64,f64,f64,f64,f64,f64,"datetime[μs, UTC]",i64
1941,2026-03-21 11:00:10.846647 UTC,"""BTC-USD""",2026-03-21 11:00:10.757698 UTC,70587.53,0.000174,"""buy""",3021,2026-03-21 11:00:11.405555 UTC,"""ETH""","""2026-03-21 11:00:00+00:00""","""2026-03-21 11:15:00+00:00""",0.0,0.57,0.41,0.59,0.43,2153.8,0.0,2026-03-21 11:00:10.404408 UTC,1
1942,2026-03-21 11:00:11.061872 UTC,"""BTC-USD""",2026-03-21 11:00:11.053426 UTC,70587.52,0.002722,"""sell""",3021,2026-03-21 11:00:11.405555 UTC,"""ETH""","""2026-03-21 11:00:00+00:00""","""2026-03-21 11:15:00+00:00""",0.0,0.57,0.41,0.59,0.43,2153.8,0.0,2026-03-21 11:00:10.404408 UTC,1
1943,2026-03-21 11:00:11.345054 UTC,"""BTC-USD""",2026-03-21 11:00:11.341167 UTC,70587.52,0.000141,"""sell""",3021,2026-03-21 11:00:11.405555 UTC,"""ETH""","""2026-03-21 11:00:00+00:00""","""2026-03-21 11:15:00+00:00""",0.0,0.57,0.41,0.59,0.43,2153.8,0.0,2026-03-21 11:00:10.404408 UTC,1
1944,2026-03-21 11:00:11.883842 UTC,"""BTC-USD""",2026-03-21 11:00:11.878472 UTC,70587.51,0.000664,"""buy""",3025,2026-03-21 11:00:12.406885 UTC,"""ETH""","""2026-03-21 11:00:00+00:00""","""2026-03-21 11:15:00+00:00""",0.0,0.57,0.41,0.59,0.43,2153.8,0.0,2026-03-21 11:00:11.405555 UTC,1
1945,2026-03-21 11:00:11.883768 UTC,"""BTC-USD""",2026-03-21 11:00:11.878472 UTC,70587.51,0.000044,"""buy""",3025,2026-03-21 11:00:12.406885 UTC,"""ETH""","""2026-03-21 11:00:00+00:00""","""2026-03-21 11:15:00+00:00""",0.0,0.57,0.41,0.59,0.43,2153.8,0.0,2026-03-21 11:00:11.405555 UTC,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8436688,2026-04-03 01:30:37.572996 UTC,"""BTC-USD""",2026-04-03 01:30:37.515937 UTC,66872.4,0.00045,"""buy""",4198208,2026-04-03 01:30:38.165010 UTC,"""BTC""","""2026-04-03 01:15:00+00:00""","""2026-04-03 01:30:00+00:00""",0.998,0.002,0.0,1.0,0.998,66783.369999,57760.08,2026-04-03 01:30:37.164514 UTC,1
8436689,2026-04-03 01:30:37.573036 UTC,"""BTC-USD""",2026-04-03 01:30:37.515937 UTC,66872.4,0.000025,"""buy""",4198208,2026-04-03 01:30:38.165010 UTC,"""BTC""","""2026-04-03 01:15:00+00:00""","""2026-04-03 01:30:00+00:00""",0.998,0.002,0.0,1.0,0.998,66783.369999,57760.08,2026-04-03 01:30:37.164514 UTC,1
8436690,2026-04-03 01:30:37.941328 UTC,"""BTC-USD""",2026-04-03 01:30:37.936154 UTC,66863.78,0.0014183,"""sell""",4198208,2026-04-03 01:30:38.165010 UTC,"""BTC""","""2026-04-03 01:15:00+00:00""","""2026-04-03 01:30:00+00:00""",0.998,0.002,0.0,1.0,0.998,66783.369999,57760.08,2026-04-03 01:30:37.164514 UTC,1


In [ ]:
btc_trades = btc_trades.with_columns(
    ((pl.col("price") * pl.col("size")) * pl.when(pl.col("side") == "buy").then(1).otherwise(-1)).alias("weighted")
)
btc_trades['weighted'].sum()


In [ ]:
clean_options = options.filter(pl.col('option_type') == 'call').drop('option_type').with_row_index('id')

In [ ]:
options.head()


id,curr_time,currency,instrument_name,expiry_datetime,strike,option_type,underlying_price,delta,mark_iv,bid_price,ask_price,mark_price,open_interest,volume
i64,str,str,str,str,f64,str,f64,f64,f64,str,str,f64,f64,f64
1120902,"""2026-03-21 16:47:26.155308+00:…","""ETH""","""ETH-22MAR26-2125-P""","""2026-03-22 00:00:00+00:00""",2125.0,"""P""",2149.8527,-0.19584,32.83,null,null,0.0015,7364.0,6292.0
1120903,"""2026-03-21 16:47:31.156183+00:…","""BTC""","""BTC-22MAR26-71000-C""","""2026-03-22 00:00:00+00:00""",71000.0,"""C""",70399.3669,0.21469,25.64,null,null,0.0013,84.9,82.9
1120904,"""2026-03-21 16:47:31.156183+00:…","""BTC""","""BTC-22MAR26-70500-C""","""2026-03-22 00:00:00+00:00""",70500.0,"""C""",70399.3669,0.4472,24.86,null,null,0.0035,222.4,134.8
1120905,"""2026-03-21 16:47:31.156183+00:…","""BTC""","""BTC-22MAR26-70000-P""","""2026-03-22 00:00:00+00:00""",70000.0,"""P""",70399.3259,-0.2967,25.84,null,null,0.002,561.6,189.5
1120906,"""2026-03-21 16:47:31.156183+00:…","""ETH""","""ETH-22MAR26-2175-C""","""2026-03-22 00:00:00+00:00""",2175.0,"""C""",2149.9038,0.19826,32.6,null,null,0.0015,1176.0,890.0


In [ ]:
kalshi.head()


id,curr_time,coin,open_time,close_time,last_price_dollars,liquidity_dollars,no_ask_dollars,no_bid_dollars,yes_ask_dollars,yes_bid_dollars,open_interest,floor_strike,cap_strike,strike_type,volume,volume_24h_fp
i64,str,str,str,str,f64,f64,f64,f64,f64,f64,str,f64,str,str,str,f64
4200710,"""2026-03-23 06:21:15.231426+00:…","""XRP""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.52,0.0,0.5,0.48,0.52,0.5,null,1.3815,null,"""greater_or_equal""",null,0.0
4200711,"""2026-03-23 06:21:15.231426+00:…","""SOL""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.58,0.0,0.51,0.47,0.53,0.49,null,86.5482,null,"""greater_or_equal""",null,0.0
4200712,"""2026-03-23 06:21:16.232547+00:…","""BTC""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.62,0.0,0.38,0.37,0.63,0.62,null,68569.81,null,"""greater_or_equal""",null,0.0
4200713,"""2026-03-23 06:21:16.232547+00:…","""ETH""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.55,0.0,0.5,0.46,0.54,0.5,null,2057.469999,null,"""greater_or_equal""",null,0.0
4200714,"""2026-03-23 06:21:16.232547+00:…","""XRP""","""2026-03-23 06:15:00+00:00""","""2026-03-23 06:30:00+00:00""",0.52,0.0,0.5,0.48,0.52,0.5,null,1.3815,null,"""greater_or_equal""",null,0.0


In [ ]:
polymarket.head()

id,curr_time,coin,time_horizon_minutes,interval_start_unix,market_slug,condition_id,strike_price,end_date,liquidity,volume,open_interest,yes_implied_price,no_implied_price,yes_buy_price,yes_sell_price,no_buy_price,no_sell_price
i64,str,str,i64,i64,str,str,f64,str,f64,f64,str,f64,f64,f64,f64,f64,f64
0,"""2026-03-28 18:00:42.725060+00:…","""ETH""",15,1774720800,"""eth-updown-15m-1774720800""","""0xd9b38938db1e4c03d96213a63a73…",2019.45,"""2026-03-28 18:15:00+00:00""",38862.5508,155.27818,null,0.54,0.46,0.47,0.48,0.52,0.53
1,"""2026-03-28 18:00:42.725060+00:…","""XRP""",15,1774720800,"""xrp-updown-15m-1774720800""","""0xf49dd149e33579a84c9ea9dfbaad…",1.3462,"""2026-03-28 18:15:00+00:00""",34937.5142,63.995962,null,0.575,0.425,0.45,0.46,0.54,0.55
2,"""2026-03-28 18:00:42.725060+00:…","""SOL""",15,1774720800,"""sol-updown-15m-1774720800""","""0x8be6b4663170642c83a1d69fe5a9…",83.17,"""2026-03-28 18:15:00+00:00""",36528.5169,42.22,null,0.535,0.465,0.49,0.5,0.5,0.51
3,"""2026-03-28 18:00:43.735854+00:…","""BTC""",15,1774720800,"""btc-updown-15m-1774720800""","""0xe2debe513a849862fee606de1680…",66853.8,"""2026-03-28 18:15:00+00:00""",48924.7393,817.438353,null,0.475,0.525,0.54,0.55,0.45,0.46
4,"""2026-03-28 18:00:43.735854+00:…","""ETH""",15,1774720800,"""eth-updown-15m-1774720800""","""0xd9b38938db1e4c03d96213a63a73…",2019.45,"""2026-03-28 18:15:00+00:00""",38862.5508,155.27818,null,0.54,0.46,0.47,0.48,0.52,0.53
